In [1]:
%pip install -q faiss-gpu

%pip install -q -U sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 2.8 MB/s eta 0:00:00a 0:00:01m


In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset

# dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")
dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

dataset["train"]

Dataset({
    features: ['doc_id', 'link', 'title', 'text', 'query'],
    num_rows: 4600
})

In [3]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanumchat-retriever-bge-m3")

train_split = dataset["train"]
val_split = dataset["train"]

corpus_texts = list(dict.fromkeys(train_split["text"])) + list(dict.fromkeys(val_split["text"]))
corpus_dataset = Dataset.from_dict({"text": corpus_texts})

corpus_embeddings = retriever.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Batches:   0%|          | 0/58 [00:00<?, ?it/s]

In [4]:
from sentence_transformers.util import mine_hard_negatives
from datasets import Dataset

train_dataset = dataset["train"].select_columns(["query", "text"])
train_dataset = train_dataset.rename_columns({
    "query": "anchor",
    "text": "positive"
})

hard_train_dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,

    range_min=1,
    range_max=100,
    relative_margin=0.05,
    sampling_strategy="top",
    batch_size=128,
    
    use_faiss=True,
    
    output_format="labeled-pair",
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Found 4582 unique queries out of 4600 total queries.
Found an average of 1.004 positives per query.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00, 10.57it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           4,600         18,239               
Mean           0.6689         0.4311         0.2429
Median         0.6845         0.4276         0.2447
Std            0.1307         0.0894         0.1109
Min           -0.0641         0.1344         0.0079
25%            0.6147         0.3701         0.1621
50%            0.6845         0.4276         0.2447
75%            0.7554         0.4903         0.3218
Max            0.9432         0.8174         0.6174
Skipped 11,446 potential negatives (2.43%) due to the relative_margin of 0.05.
Could not find enough negatives for 157 samples (0.68%). Consider adjusting the range_max, range_min and relative_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4600
Enriched dataset size: 22771
{'anchor': 'how to learn spells', 'positive': 'Spell Colleges. See also. Spell Colleges. See also. Magick - for info abo

In [5]:
val_dataset = dataset["validation"].select_columns(["query", "text"])
val_dataset = val_dataset.rename_columns({
    "query": "anchor",
    "text": "positive"
})

hard_val_dataset = mine_hard_negatives(
    dataset=val_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,

    range_min=1,
    range_max=100,
    relative_margin=0.05,
    sampling_strategy="top",
    batch_size=128,
    
    use_faiss=True,
    
    output_format="labeled-pair",
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00, 522.13it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count             574          2,252               
Mean           0.5787         0.4090         0.1789
Median         0.6028         0.4031         0.1670
Std            0.1748         0.0970         0.1189
Min           -0.0285         0.1516         0.0099
25%            0.4616         0.3436         0.0775
50%            0.6031         0.4032         0.1670
75%            0.7117         0.4721         0.2600
Max            0.9404         0.7290         0.5887
Skipped 3,439 potential negatives (5.93%) due to the relative_margin of 0.05.
Could not find enough negatives for 44 samples (1.53%). Consider adjusting the range_max, range_min and relative_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4600
Enriched dataset size: 22771
{'anchor': 'how to learn spells', 'positive': 'Spell Colleges. See also. Spell Colleges. See also. Magick - for info about

In [6]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    hard_val_dataset["anchor"], hard_val_dataset["positive"], hard_val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
hard_eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(hard_eval_samples)}")

Evaluation queries: 563


In [7]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=hard_eval_samples,
    at_k=10,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [8]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [9]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_train_dataset],
    "document": [p["positive"] for p in hard_train_dataset],
    "label": [p["label"] for p in hard_train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_val_dataset],
    "document": [p["positive"] for p in hard_val_dataset],
    "label": [p["label"] for p in hard_val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

In [10]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    warmup_steps=0.1,
    lr_scheduler_type="linear",

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps = 2,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@10",
    greater_is_better=True,

    eval_on_start=True,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,

    evaluator=evaluator,
    loss=train_loss,
    # callbacks=[
    #         EarlyStoppingCallback(
    #                 early_stopping_patience=3,
    #                 early_stopping_threshold=0.001,
    #             )
    #     ],
)

In [11]:
trainer.train()

Step,Training Loss,Validation Loss,Val-hard-negatives Map,Val-hard-negatives Mrr@10,Val-hard-negatives Ndcg@10
0,No log,0.298096,0.955003,0.955003,0.966492
100,0.294086,0.213523,0.961220,0.961220,0.971206
200,0.259711,0.206392,0.966696,0.966696,0.975305
300,0.249655,0.183390,0.974245,0.974245,0.980947
400,0.221664,0.194588,0.969953,0.969953,0.977737
500,0.224612,0.183741,0.976021,0.976021,0.982202
600,0.204956,0.188544,0.979870,0.979870,0.985057
700,0.210743,0.161749,0.979426,0.979426,0.984715
712,0.210743,0.160899,0.979426,0.979426,0.984715


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

TrainOutput(global_step=712, training_loss=0.23754085985462317, metrics={'train_runtime': 3713.0585, 'train_samples_per_second': 6.133, 'train_steps_per_second': 0.192, 'total_flos': 0.0, 'train_loss': 0.23754085985462317, 'epoch': 1.0})

In [12]:
model.push_to_hub("pameydorke/arcanumchat-reranker-bge-reranker-v2-m3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/pameydorke/arcanumchat-reranker-bge-reranker-v2-m3/commit/6dadf17d1ef7e70f3d54ff2a6d36eaf849d5cc8a'

In [20]:
base_model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

ft_model = CrossEncoder("pameydorke/arcanumsearch-reranker-bge-reranker-v2-m3", num_labels=1)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/48.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

In [21]:
import math
import numpy as np

def get_per_query_ndcg(model, samples, k=10):
    """
    Computes NDCG@10 for each query in a list of samples.

    Args:
        model: A loaded CrossEncoder model.
        samples (list): List of dicts with 'query', 'positive', and 'negative' keys.
        k (int): The cutoff for NDCG.

    Returns:
        list: A list of NDCG@10 scores, one per query.
    """
    per_query_scores = []
    
    for sample in samples:
        query = sample["query"]
        positives = set(sample["positive"])
        # Combine all candidates (positives + negatives) for reranking
        candidates = list(positives) + sample.get("negative", [])
        
        # Get the cross-encoder's scores for all (query, candidate) pairs
        pairs = [(query, doc) for doc in candidates]
        scores = model.predict(pairs)
        
        # Sort candidates by score in descending order
        ranked_candidates = [doc for _, doc in sorted(zip(scores, candidates), reverse=True)]
        
        # Calculate DCG and IDCG for the top-k ranked list
        dcg = 0.0
        idcg = 0.0
        for i, doc in enumerate(ranked_candidates[:k]):
            rel = 1.0 if doc in positives else 0.0
            dcg += rel / math.log2(i + 2) # i+2 because rank starts at 1
            
        # IDCG is the score of a perfect ranking
        for i in range(min(len(positives), k)):
            idcg += 1.0 / math.log2(i + 2)
            
        ndcg = dcg / idcg if idcg > 0 else 0.0
        per_query_scores.append(ndcg)
        
    return per_query_scores

In [22]:
base_scores = np.array(get_per_query_ndcg(base_model, eval_samples))
ft_scores = np.array(get_per_query_ndcg(ft_model, eval_samples))

# Now you can proceed with the paired bootstrap
diffs = ft_scores - base_scores
rng = np.random.default_rng(42)
boot = np.array([
    diffs[rng.integers(0, len(diffs), len(diffs))].mean()
    for _ in range(10_000)
])
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Mean improvement: {diffs.mean():+.4f}")
print(f"95% Confidence Interval: [{lo:+.4f}, {hi:+.4f}]")

Mean improvement: +0.0282
95% Confidence Interval: [+0.0141, +0.0425]
